# Synapsys — Reference Notebook

## Goal

This notebook demonstrates the main features of the **synapsys** control systems library using **a single reference system** throughout every section. The central goal is to show that:

> **All ways of representing and simulating the same linear system are mathematically equivalent and produce the same numerical result.**

| Section | Content | Main API |
|---|---|---|
| 1 | Transfer Function and State-Space Model | `tf()`, `ss()` |
| 2 | **Equivalence comparison** (TF, SS, discrete, `evolve()`) | `step()`, `evolve()` |
| 3 | Discretisation methods | `c2d()` |
| 4 | Block diagram algebra | `feedback()`, `series()` |
| 5 | Frequency-domain analysis | `bode()` |
| 6 | PID closed-loop controller | `PID` |
| 7 | LQR regulator (state feedback) | `lqr()` |
| 8 | Real-time simulation with agents | `PlantAgent`, `ControllerAgent` |

---

## Reference System

Every section uses **the same underdamped second-order system**:

$$G(s) = \frac{\omega_n^2}{s^2 + 2\zeta\omega_n\,s + \omega_n^2} = \frac{25}{s^2 + 5s + 25}$$

with $\omega_n = 5\,\text{rad/s}$ and $\zeta = 0.5$. This system is stable, has unity DC gain, and exhibits a damped oscillatory response characteristic of many real physical systems (vehicle suspension, robotic arm, underdamped RLC circuit, etc.).

The equivalent state-space form (controllable canonical form) is:

$$\dot{x} = \underbrace{\begin{bmatrix}0 & 1\\-25 & -5\end{bmatrix}}_{A}x + \underbrace{\begin{bmatrix}0\\25\end{bmatrix}}_{B}u, \qquad y = \underbrace{\begin{bmatrix}1 & 0\end{bmatrix}}_{C}x$$

In [1]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.30,
        "lines.linewidth": 1.8,
        "font.size": 11,
    }
)

# ── Reference system parameters ───────────────────────────────────────────────
WN = 5.0  # natural frequency (rad/s)
ZETA = 0.5  # damping ratio

print(
    f"Reference system: G(s) = {WN**2:.0f} / (s² + {2 * ZETA * WN:.0f}s + {WN**2:.0f})"
)
print(f"  ωn = {WN} rad/s  |  ζ = {ZETA}  |  type: underdamped (ζ < 1)")

Reference system: G(s) = 25 / (s² + 5s + 25)
  ωn = 5.0 rad/s  |  ζ = 0.5  |  type: underdamped (ζ < 1)


---
## 1. Transfer Function and State-Space Model

The **transfer function** $G(s)$ and the **state-space** representation $(A, B, C, D)$ are two different representations of the same linear dynamic system.

- `tf(num, den)` builds the model from numerator and denominator polynomial coefficients.
- `ss(A, B, C, D)` builds the model from the system matrices.

Both expose the same interface: `.poles()`, `.is_stable()`, `.evaluate(s)` and are interchangeable in `step()`, `bode()` and `c2d()`.

In [2]:
from synapsys.api import bode, c2d, feedback, ss, step, tf

# ── 1a. Transfer Function ─────────────────────────────────────────────────────
G_tf = tf([WN**2], [1, 2 * ZETA * WN, WN**2])

print("=== Transfer Function ===")
print(G_tf)
print(f"  Poles  : {G_tf.poles()}")
print(f"  Zeros  : {G_tf.zeros()}")
print(f"  DC gain (s→0) : {G_tf.evaluate(0):.4f}")
print(f"  Stable : {G_tf.is_stable()}")

print()

# ── 1b. State-Space (controllable canonical form) ─────────────────────────────
A = np.array([[0.0, 1.0], [-(WN**2), -2 * ZETA * WN]])
B = np.array([[0.0], [WN**2]])
C = np.array([[1.0, 0.0]])
D = np.array([[0.0]])

G_ss = ss(A, B, C, D)

print("=== State-Space ===")
print(G_ss)
print(f"  Poles    : {G_ss.poles()}")
print(f"  n_states : {G_ss.n_states}")
print(f"  Stable   : {G_ss.is_stable()}")

print()
print("Check: are the poles identical in both representations?")
poles_match = np.allclose(np.sort_complex(G_tf.poles()), np.sort_complex(G_ss.poles()))
print(f"  Poles match: {poles_match}  ✓" if poles_match else "  MISMATCH in poles!")

=== Transfer Function ===
TransferFunction(num=[25.0], den=[1.0, 5.0, 25.0], continuous)
  Poles  : [-2.5+4.33012702j -2.5-4.33012702j]
  Zeros  : []
  DC gain (s→0) : 1.0000+0.0000j
  Stable : True

=== State-Space ===
StateSpace(n_states=2, n_inputs=1, n_outputs=1, continuous)
  Poles    : [-2.5+4.33012702j -2.5-4.33012702j]
  n_states : 2
  Stable   : True

Check: are the poles identical in both representations?
  Poles match: True  ✓


---
## 2. Representation Equivalence — all methods produce the same result

This is the central section of the notebook. We compute the step response of the **same system** using four distinct approaches:

| Method | How it works |
|---|---|
| `step(G_tf)` | Continuous TF integration via SciPy (`lsim` with unit step) |
| `step(G_ss)` | Continuous SS integration via SciPy (`lsim` with unit step) |
| `step(G_tf_d)` | Step response of the **discrete TF** model (ZOH) |
| `evolve()` manual | Sample-by-sample loop on the **discrete SS** model (ZOH) |

Continuous methods should agree up to machine precision. Discrete methods introduce a **discretisation error** proportional to $\Delta t$; the smaller $\Delta t$, the closer to the continuous result.

> **Why does this matter?** In control engineering you can switch representation at any time without losing fidelity. Choosing TF vs SS is a matter of convenience, not accuracy.

In [3]:
DT_CMP = 0.01  # 100 Hz — discrete simulation resolution

# ── Method 1: step() on continuous TF ────────────────────────────────────────
t_tf, y_tf = step(G_tf)

# ── Method 2: step() on continuous SS ────────────────────────────────────────
t_ss, y_ss = step(G_ss)

# ── Method 3: step() on discretised TF (ZOH) ─────────────────────────────────
G_tf_d = c2d(G_tf, dt=DT_CMP, method="zoh")
N_cmp = int(t_tf[-1] / DT_CMP) + 1
t_tfd, y_tfd = step(G_tf_d, n=N_cmp)

# ── Method 4: manual evolve() loop on discrete SS (ZOH) ──────────────────────
G_ss_d = c2d(G_ss, dt=DT_CMP, method="zoh")
x_man = np.zeros(G_ss_d.n_states)  # zero initial state
y_man = np.zeros(N_cmp)

for k in range(N_cmp):
    x_man, y_k = G_ss_d.evolve(x_man, np.array([1.0]))  # unit step u=1
    y_man[k] = float(y_k[0])

t_man = np.arange(N_cmp) * DT_CMP

# ── Comparison plot ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(t_tf, y_tf, lw=3.0, color="steelblue", label="step(G_tf)  — continuous TF")
ax.plot(
    t_ss,
    y_ss,
    lw=2.0,
    color="darkorange",
    linestyle="--",
    label="step(G_ss)  — continuous SS",
)
ax.step(
    t_tfd,
    y_tfd,
    lw=1.6,
    color="seagreen",
    where="post",
    linestyle=":",
    label=f"step(G_tf_d)  — discrete TF ZOH  Δt={DT_CMP}s",
)
ax.plot(
    t_man,
    y_man,
    lw=1.2,
    color="crimson",
    linestyle="-.",
    label=f"evolve() manual — discrete SS ZOH  Δt={DT_CMP}s",
)
ax.axhline(
    1.0, color="k", linestyle=":", lw=1.2, alpha=0.4, label="Unit step reference"
)

ax.set_title("Representation Equivalence — Step Response", fontsize=13)
ax.set_xlabel("Time (s)")
ax.set_ylabel("y(t)")
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()

# ── Numerical error check ─────────────────────────────────────────────────────
y_ss_i = np.interp(t_tf, t_ss, y_ss)
y_tfd_i = np.interp(t_tf, t_tfd, y_tfd)
y_man_i = np.interp(t_tf, t_man, y_man)

print("Maximum error relative to continuous TF:")
print(
    f"  continuous TF  vs  continuous SS    : {np.max(np.abs(y_tf - y_ss_i)):.2e}  ← machine precision"
)
print(
    f"  continuous TF  vs  discrete TF ZOH : {np.max(np.abs(y_tf - y_tfd_i)):.5f}   ← discretisation error (Δt={DT_CMP}s)"
)
print(
    f"  continuous TF  vs  manual evolve()  : {np.max(np.abs(y_tf - y_man_i)):.5f}   ← identical to previous"
)
print()
print("Conclusion: continuous TF and SS are numerically identical.")
print(
    "            manual evolve() and discrete step() are equivalent (same internal structure)."
)

Maximum error relative to continuous TF:
  continuous TF  vs  continuous SS    : 1.55e-15  ← machine precision
  continuous TF  vs  discrete TF ZOH : 0.00020   ← discretisation error (Δt=0.01s)
  continuous TF  vs  manual evolve()  : 0.00020   ← identical to previous

Conclusion: continuous TF and SS are numerically identical.
            manual evolve() and discrete step() are equivalent (same internal structure).


/home/osfarias/workspace/workspace_mestrado/pycontrol/.venv/lib/python3.12/site-packages/scipy/signal/_ltisys.py:603: BadCoefficients: Badly conditioned filter coefficients (numerator): the results may be meaningless
  self.num, self.den = normalize(*system)
/tmp/ipykernel_3115050/1491929467.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Discretisation Methods

`c2d(sys, dt, method)` converts a continuous model to discrete. synapsys supports four methods:

| Method | Description | When to use |
|---|---|---|
| `'zoh'` | Zero-Order Hold — exact for piecewise-constant inputs | Default; best fidelity |
| `'bilinear'` | Tustin transform $s = \frac{2}{\Delta t}\frac{z-1}{z+1}$ | Preserves Bode shape |
| `'euler'` | Forward finite differences $s = \frac{z-1}{\Delta t}$ | Simple, less accurate |
| `'backward_diff'` | Backward finite differences $s = \frac{z-1}{z\,\Delta t}$ | Always stable if continuous model is stable |

The choice of method and step size $\Delta t$ affects accuracy. Here we compare all methods against the continuous reference response.

In [4]:
DT_C2D = 0.05  # 20 Hz — larger step to highlight differences between methods
N_c2d = int(3.0 / DT_C2D) + 1

methods = ["zoh", "bilinear", "euler", "backward_diff"]
colors = ["seagreen", "darkorange", "crimson", "mediumpurple"]

fig, ax = plt.subplots(figsize=(11, 5))

# Continuous reference
t_ref, y_ref = step(G_tf)
ax.plot(t_ref, y_ref, lw=2.5, color="steelblue", label="Continuous G(s) — reference")

# Discrete models
for method, color in zip(methods, colors):
    G_d = c2d(G_tf, dt=DT_C2D, method=method)
    t_d, y_d = step(G_d, n=N_c2d)
    ax.step(
        t_d,
        y_d,
        where="post",
        lw=1.5,
        color=color,
        linestyle="--",
        label=f"{method}  (Δt={DT_C2D}s)",
    )

ax.axhline(1.0, color="k", linestyle=":", lw=1.0, alpha=0.35)
ax.set_title(f"Discretisation Methods Comparison  (Δt = {DT_C2D}s)", fontsize=13)
ax.set_xlabel("Time (s)")
ax.set_ylabel("y(t)")
ax.set_xlim(0, 3.0)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

# ── Discretisation error table ────────────────────────────────────────────────
print(f"Maximum error per method relative to continuous  (Δt = {DT_C2D}s):")
for method in methods:
    G_d = c2d(G_tf, dt=DT_C2D, method=method)
    t_d, y_d = step(G_d, n=N_c2d)
    y_d_i = np.interp(t_ref, t_d, y_d)
    err = np.max(np.abs(y_ref - y_d_i))
    print(f"  {method:<15}: {err:.5f}")

print()
print("Note: ZOH and bilinear are more accurate than Euler methods for the same Δt.")

Maximum error per method relative to continuous  (Δt = 0.05s):
  zoh            : 0.00668
  bilinear       : 0.06457
  euler          : 0.11678
  backward_diff  : 0.13623

Note: ZOH and bilinear are more accurate than Euler methods for the same Δt.


/tmp/ipykernel_3115050/3444287884.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Block Diagram Algebra

synapsys supports direct algebraic operations on LTI models:

- **Series:** `G1 * G2`  or  `series(G1, G2)` → $G_1(s)\cdot G_2(s)$
- **Parallel:** `G1 + G2`  or  `parallel(G1, G2)` → $G_1(s) + G_2(s)$
- **Closed-loop:** `feedback(L)` → $T(s) = \dfrac{L(s)}{1 + L(s)}$ (unity negative feedback)

### Standard topology (controller in the forward path)

```
r ──▶[+]──▶[ Ctrl ]──▶[ G ]──▶ y
      ▲─────────────────────────┘
               (−)
```

The closed-loop transfer function is:

$$T(s) = \frac{C(s)\,G(s)}{1 + C(s)\,G(s)}$$

### Steady-state error with a proportional controller

With $C(s) = K_p$ (no integrator), the **steady-state error for a step input** is:

$$e_{ss} = \frac{1}{1 + K_p\,G(0)} \neq 0$$

Zero error requires at least one integrator in the open loop (PI controller or type-1 plant). This is a direct consequence of the **Final Value Theorem**.

In [5]:
from synapsys.api import series

# ── Proportional controller Ctrl(s) = Kp ─────────────────────────────────────
Kp = 2.0
Ctrl = tf([Kp], [1])  # pure gain

# ── Open loop: L(s) = Ctrl(s)·G(s)  (controller in forward path) ─────────────
L = series(Ctrl, G_tf)

# ── Closed loop: T(s) = L(s)/(1+L(s))  (unity negative feedback) ─────────────
T = feedback(L)

print("=== Open Loop  L(s) = Ctrl(s)·G(s) ===")
print(L)
print(f"  Poles: {L.poles()}")
print(f"  DC gain: {L.evaluate(0):.4f}")

print()
print("=== Closed Loop  T(s) = L(s)/(1+L(s)) ===")
print(T)
dc_T = T.evaluate(0).real
e_ss = 1 - dc_T
print(f"  Poles: {T.poles()}")
print(f"  DC gain: {dc_T:.4f}")
print(f"  Steady-state error (unit step): e_ss = 1/(1+Kp·G(0)) = {e_ss:.4f}")
print()
print(
    f"  Analytical check: 1/(1 + {Kp}×{G_tf.evaluate(0).real:.1f}) = {1 / (1 + Kp * G_tf.evaluate(0).real):.4f}"
)
print()
print("  ─── Why is there steady-state error? ────────────────────────────────")
print("  With C(s)=Kp (no integrator), the open loop L(s) is type 0.")
print("  By the Final Value Theorem: e_ss = 1/(1+L(0))")
print("  For zero error: add an integrator (PI controller) or use PID/LQR.")

# ── Plots: step responses for different Kp values ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Step response — plant vs closed-loop for different Kp
t_plant, y_plant = step(G_tf)
ax1.plot(t_plant, y_plant, lw=2.5, color="steelblue", label="G(s) — plant (open loop)")

kp_vals = [1.0, 2.0, 5.0, 10.0]
kp_colors = ["gold", "darkorange", "tomato", "crimson"]

for kp_i, col in zip(kp_vals, kp_colors):
    T_i = feedback(series(tf([kp_i], [1]), G_tf))
    t_i, y_i = step(T_i)
    dc_i = T_i.evaluate(0).real
    e_i = 1 - dc_i
    ax1.plot(t_i, y_i, lw=1.8, color=col, label=f"Kp={kp_i:.0f}  →  e_ss={e_i:.3f}")

ax1.axhline(1.0, color="k", linestyle=":", lw=1.0, alpha=0.4, label="Reference = 1")
ax1.set_title("Step Response — Effect of Proportional Gain", fontsize=12)
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("y(t)")
ax1.legend(fontsize=8)

# Steady-state error vs Kp (analytical curve)
kp_range = np.linspace(0.1, 20, 200)
ess_range = 1.0 / (1.0 + kp_range * G_tf.evaluate(0).real)
ax2.plot(kp_range, ess_range, lw=2.0, color="steelblue")
ax2.axhline(
    0,
    color="k",
    linestyle="--",
    lw=1.0,
    alpha=0.5,
    label="Zero error (impossible with P only)",
)
ax2.scatter(
    kp_vals,
    [1 / (1 + k * G_tf.evaluate(0).real) for k in kp_vals],
    color=kp_colors,
    s=80,
    zorder=5,
)
ax2.set_title("Steady-State Error vs Kp  (proportional controller)", fontsize=12)
ax2.set_xlabel("Kp")
ax2.set_ylabel("e_ss")
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

=== Open Loop  L(s) = Ctrl(s)·G(s) ===
TransferFunction(num=[50.0], den=[1.0, 5.0, 25.0], continuous)
  Poles: [-2.5+4.33012702j -2.5-4.33012702j]
  DC gain: 2.0000+0.0000j

=== Closed Loop  T(s) = L(s)/(1+L(s)) ===
TransferFunction(num=[50.0], den=[1.0, 5.0, 75.0], continuous)
  Poles: [-2.5+8.29156198j -2.5-8.29156198j]
  DC gain: 0.6667
  Steady-state error (unit step): e_ss = 1/(1+Kp·G(0)) = 0.3333

  Analytical check: 1/(1 + 2.0×1.0) = 0.3333

  ─── Why is there steady-state error? ────────────────────────────────
  With C(s)=Kp (no integrator), the open loop L(s) is type 0.
  By the Final Value Theorem: e_ss = 1/(1+L(0))
  For zero error: add an integrator (PI controller) or use PID/LQR.


/tmp/ipykernel_3115050/2066851238.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Frequency-Domain Analysis — Bode Diagram

The Bode diagram shows how the system responds to sinusoidal inputs at different frequencies:

- **Magnitude:** gain $|G(j\omega)|$ in dB — how much the output amplitude is amplified or attenuated.
- **Phase:** phase shift $\angle G(j\omega)$ in degrees — the lag or lead of the output relative to the input.

`bode(sys)` returns `(ω, magnitude_dB, phase_deg)` — same convention as MATLAB.

For the reference system we expect:
- Resonance peak near $\omega \approx \omega_n\sqrt{1-2\zeta^2}$.
- Phase of $-90°$ at $\omega = \omega_n$.
- $-40\,\text{dB/dec}$ roll-off for $\omega \gg \omega_n$ (second-order system).

We overlay TF and SS to confirm the diagrams are identical.

In [6]:
w_tf, mag_tf, phase_tf = bode(G_tf)
w_ss, mag_ss, phase_ss = bode(G_ss)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Magnitude
ax1.semilogx(w_tf, mag_tf, lw=2.5, color="steelblue", label="G_tf — Transfer Function")
ax1.semilogx(
    w_ss, mag_ss, lw=1.5, color="darkorange", linestyle="--", label="G_ss — State-Space"
)
ax1.axvline(WN, color="k", linestyle=":", lw=1.0, alpha=0.5, label=f"ωn = {WN} rad/s")
ax1.set_ylabel("Magnitude (dB)")
ax1.set_title(r"Bode Diagram — $G(s) = 25/(s^2+5s+25)$", fontsize=13)
ax1.legend(fontsize=9)

# Phase
ax2.semilogx(w_tf, phase_tf, lw=2.5, color="steelblue", label="G_tf")
ax2.semilogx(w_ss, phase_ss, lw=1.5, color="darkorange", linestyle="--", label="G_ss")
ax2.axvline(WN, color="k", linestyle=":", lw=1.0, alpha=0.5)
ax2.axhline(-90, color="gray", linestyle=":", lw=0.8, alpha=0.5, label="-90° at ω=ωn")
ax2.set_ylabel("Phase (degrees)")
ax2.set_xlabel("Frequency (rad/s)")
ax2.legend(fontsize=9)

fig.tight_layout()
plt.show()

# ── Numerical check ───────────────────────────────────────────────────────────
err_bode = np.max(np.abs(mag_tf - np.interp(w_tf, w_ss, mag_ss)))
print(f"Maximum magnitude error between TF and SS: {err_bode:.2e}  ← machine precision")

/home/osfarias/workspace/workspace_mestrado/pycontrol/.venv/lib/python3.12/site-packages/scipy/signal/_filter_design.py:1233: BadCoefficients: Badly conditioned filter coefficients (numerator): the results may be meaningless
  b, a = normalize(b, a)


Maximum magnitude error between TF and SS: 1.42e-14  ← machine precision


/tmp/ipykernel_3115050/1339387140.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. PID Closed-Loop Controller

The synapsys discrete `PID` implements the velocity-form algorithm with:
- **Anti-windup** via back-calculation (prevents integrator saturation).
- Configurable **output saturation** (`u_min`, `u_max`).

We manually simulate the control loop at each time step using the **same reference plant** discretised at 100 Hz:

```
┌──────────┐    u(k)    ┌──────────┐    y(k)
│   PID    │──────────▶ │  Plant   │──────────▶
└──────────┘            └──────────┘
      ▲                                  │
      │         e(k) = r(k) − y(k)       │
      └─────────────────────────────────-┘
```

The reference (setpoint) is a **sinusoid** $r(t) = 3 + 1.5\sin(2\pi \cdot 0.3\, t)$ to make the dynamic response more informative.

In [7]:
from synapsys.algorithms import PID

DT_PID = 0.01  # 100 Hz
T_PID = 8.0  # simulation duration (seconds)
N_PID = int(T_PID / DT_PID)

# ── Plant: G(s) discretised with ZOH ─────────────────────────────────────────
plant_d = c2d(G_ss, dt=DT_PID, method="zoh")

# ── PID tuned for the reference system ───────────────────────────────────────
# Parameters chosen empirically for a good transient response
pid = PID(Kp=1.8, Ki=2.5, Kd=0.05, dt=DT_PID, u_min=-20.0, u_max=20.0)


# ── Sinusoidal reference: r(t) = 3 + 1.5·sin(2π·0.3·t) ──────────────────────
def setpoint(t):
    return 3.0 + 1.5 * np.sin(2 * np.pi * 0.3 * t)


# ── Simulation loop ───────────────────────────────────────────────────────────
x = np.zeros(plant_d.n_states)  # zero initial state
ts, ys, us, rs = [], [], [], []

for k in range(N_PID):
    t_k = k * DT_PID
    r_k = setpoint(t_k)

    # Current plant output  (C·x, C shape (1,n), x shape (n,) → scalar)
    y_k = float(np.dot(plant_d.C[0], x))

    # Control action
    u_k = pid.compute(setpoint=r_k, measurement=y_k)

    # State evolution
    x, _ = plant_d.evolve(x, np.array([u_k]))

    ts.append(t_k)
    ys.append(y_k)
    us.append(u_k)
    rs.append(r_k)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(
    ts,
    rs,
    lw=1.5,
    color="k",
    linestyle="--",
    alpha=0.7,
    label="r(t) — sinusoidal reference",
)
ax1.plot(ts, ys, lw=2.0, color="steelblue", label="y(t) — plant output")
ax1.set_ylabel("y(t)")
ax1.set_title(
    f"PID Closed-Loop — G(s) = 25/(s²+5s+25)  |  Kp={pid.Kp}, Ki={pid.Ki}, Kd={pid.Kd}",
    fontsize=12,
)
ax1.legend(fontsize=10)

ax2.plot(ts, us, lw=1.8, color="darkorange", label="u(t) — control signal")
ax2.axhline(20.0, color="r", linestyle=":", lw=1.0, alpha=0.5, label="u_max / u_min")
ax2.axhline(-20.0, color="r", linestyle=":", lw=1.0, alpha=0.5)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("u(t)")
ax2.legend(fontsize=10)

fig.tight_layout()
plt.show()

# ── Performance metrics ───────────────────────────────────────────────────────
erro = np.array(rs) - np.array(ys)
print(f"RMS error (steady state, t > 2s): {np.sqrt(np.mean(erro[200:] ** 2)):.4f}")
print(f"Peak absolute error             : {np.max(np.abs(erro)):.4f}")

RMS error (steady state, t > 2s): 0.3493
Peak absolute error             : 3.0037


/tmp/ipykernel_3115050/4070776239.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. LQR Regulator — Reference Tracking

By design, LQR **regulates to zero**: it minimises the deviation of the state from the origin. To track a **non-zero reference** $r \neq 0$, we use a feedforward pre-compensation gain $\bar{N}$ (the "Nbar" method):

$$u(t) = -K x(t) + \bar{N}\, r$$

The gain $\bar{N}$ is chosen so that $y_{ss} = r$ at steady state. Derivation:

$$\text{At steady state: } 0 = (A - BK)\,x_{ss} + B\bar{N}r \implies x_{ss} = -(A-BK)^{-1}B\bar{N}r$$

$$y_{ss} = C_{\text{obs}}\,x_{ss} = r \implies \bar{N} = \frac{-1}{C_{\text{obs}}\,(A-BK)^{-1}B}$$

Control diagram with $\bar{N}$:

```
         ┌────────────────────────────────────────┐
r ──Nbar─┤+─▶[ u = -Kx + Nbar·r ]──▶[ Plant ]──▶ y
         │         ▲                        │
         │         │     x (states)         │
         └─────────┴────────────────────────┘
```

**Important:** $\bar{N}$ guarantees zero steady-state error only for **constant** references (step). For time-varying references (ramp, sinusoid) integral action on the states is required.

In [8]:
from synapsys.algorithms import lqr

# Output observation row: y = x₁  (first row of C = [[1, 0]])
C_obs = np.array([1.0, 0.0])

print("Reference system matrices:")
print(f"  A =\n{A}")
print(f"  B = {B.flatten()}")
print(f"  C_obs = {C_obs}  (y = x₁)")

# ── LQR weights ───────────────────────────────────────────────────────────────
Q = np.diag([20.0, 1.0])  # penalise x1 (output) more than x2 (derivative)
R = np.array([[1.0]])

K, P = lqr(A, B, Q, R)

print(f"\nOptimal gain K = {K}")
print(f"Cost matrix P =\n{P}")

# ── Closed-loop poles ─────────────────────────────────────────────────────────
A_cl = A - B @ K
cl_poles = np.linalg.eigvals(A_cl)
print(f"\nOpen-loop poles  : {np.linalg.eigvals(A)}")
print(f"Closed-loop poles: {cl_poles}  (LQR shifted left)")
print(f"All Re(p) < 0 (stable): {np.all(np.real(cl_poles) < 0)}")

# ── Feedforward pre-compensation gain Nbar ────────────────────────────────────
# Ensures y_ss = r for constant reference
# Nbar = -1 / (C_obs · inv(A_cl) · B)
Nbar = float(-1.0 / (C_obs @ np.linalg.solve(A_cl, B.flatten())))
print(f"\nPre-compensation gain  Nbar = {Nbar:.4f}")
print("  Control law: u(t) = -K·x(t) + Nbar·r")
print("  Check: y_ss = C·inv(A-BK)·(-B·Nbar·r) → should → r  ✓")

Reference system matrices:
  A =
[[  0.   1.]
 [-25.  -5.]]
  B = [ 0. 25.]
  C_obs = [1. 0.]  (y = x₁)

Optimal gain K = [[3.58257569 0.95178386]]
Cost matrix P =
[[5.07813671 0.14330303]
 [0.14330303 0.03807135]]

Open-loop poles  : [-2.5+4.33012702j -2.5-4.33012702j]
Closed-loop poles: [ -4.76828977 -24.02630668]  (LQR shifted left)
All Re(p) < 0 (stable): True

Pre-compensation gain  Nbar = 4.5826
  Control law: u(t) = -K·x(t) + Nbar·r
  Check: y_ss = C·inv(A-BK)·(-B·Nbar·r) → should → r  ✓


In [9]:
# ── Simulation: pure regulation (r=0) vs reference tracking with Nbar ─────────
DT_LQR = 0.005  # 200 Hz
T_LQR = 5.0
steps = int(T_LQR / DT_LQR)

SETPOINT_LQR = 2.0  # non-zero reference

# Case 1: pure regulation (no Nbar) — x0 = [2, 0], reference = 0
x_reg = np.array([2.0, 0.0])
# Case 2: reference tracking with Nbar — x0 = [0, 0], reference = 2.0
x_trk = np.zeros(2)

t_hist = []
y_reg_hist, u_reg_hist = [], []
y_trk_hist, u_trk_hist = [], []

for k in range(steps):
    t_k = k * DT_LQR

    # ── Case 1: regulation (u = -Kx) ─────────────────────────────────────────
    u_reg = float(-(K @ x_reg).flat[0])
    y_reg = float(np.dot(C_obs, x_reg))
    x_reg = x_reg + DT_LQR * (A @ x_reg + B.flatten() * u_reg)

    # ── Case 2: tracking (u = -Kx + Nbar·r) ──────────────────────────────────
    u_trk = float(-(K @ x_trk).flat[0]) + Nbar * SETPOINT_LQR
    y_trk = float(np.dot(C_obs, x_trk))
    x_trk = x_trk + DT_LQR * (A @ x_trk + B.flatten() * u_trk)

    t_hist.append(t_k)
    y_reg_hist.append(y_reg)
    u_reg_hist.append(u_reg)
    y_trk_hist.append(y_trk)
    u_trk_hist.append(u_trk)

t_arr = np.array(t_hist)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("LQR — Regulation to Zero  vs  Reference Tracking (Nbar)", fontsize=13)

# ── Outputs ───────────────────────────────────────────────────────────────────
axes[0, 0].plot(t_arr, y_reg_hist, lw=2.0, color="steelblue", label="y(t) — regulation")
axes[0, 0].axhline(0, color="k", linestyle="--", lw=1.0, alpha=0.5, label="r = 0")
axes[0, 0].set_title("Regulation: x₀=[2,0] → 0")
axes[0, 0].set_ylabel("y(t)")
axes[0, 0].legend(fontsize=9)

axes[0, 1].plot(t_arr, y_trk_hist, lw=2.0, color="darkorange", label="y(t) — tracking")
axes[0, 1].axhline(
    SETPOINT_LQR,
    color="k",
    linestyle="--",
    lw=1.0,
    alpha=0.5,
    label=f"r = {SETPOINT_LQR}",
)
axes[0, 1].set_title(f"Tracking: x₀=[0,0] → r={SETPOINT_LQR}  (Nbar={Nbar:.3f})")
axes[0, 1].set_ylabel("y(t)")
axes[0, 1].legend(fontsize=9)

# ── Control signals ───────────────────────────────────────────────────────────
axes[1, 0].plot(t_arr, u_reg_hist, lw=2.0, color="seagreen", label="u(t) = −Kx")
axes[1, 0].axhline(0, color="k", linestyle="--", lw=1.0, alpha=0.4)
axes[1, 0].set_ylabel("u(t)")
axes[1, 0].set_xlabel("Time (s)")
axes[1, 0].legend(fontsize=9)

axes[1, 1].plot(
    t_arr, u_trk_hist, lw=2.0, color="mediumpurple", label=f"u(t) = −Kx + {Nbar:.2f}·r"
)
axes[1, 1].axhline(0, color="k", linestyle="--", lw=1.0, alpha=0.4)
axes[1, 1].set_ylabel("u(t)")
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].legend(fontsize=9)

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

# ── Steady-state error check ──────────────────────────────────────────────────
y_ss_reg = y_reg_hist[-1]
y_ss_trk = y_trk_hist[-1]
print(f"Regulation  — y_ss = {y_ss_reg:.6f}  (expected: 0.0)")
print(f"Tracking    — y_ss = {y_ss_trk:.6f}  (expected: {SETPOINT_LQR})")
print(f"  Steady-state error: {abs(SETPOINT_LQR - y_ss_trk):.2e}  ← virtually zero ✓")
print(
    f"\nCost J (regulation)  ≈ {np.sum(np.array(y_reg_hist) ** 2 * 20 + np.array(u_reg_hist) ** 2) * DT_LQR:.4f}"
)
print(
    f"Cost J (tracking)    ≈ {np.sum((np.array(y_trk_hist) - SETPOINT_LQR) ** 2 * 20 + np.array(u_trk_hist) ** 2) * DT_LQR:.4f}"
)

Regulation  — y_ss = 0.000000  (expected: 0.0)
Tracking    — y_ss = 2.000000  (expected: 2.0)
  Steady-state error: 8.47e-11  ← virtually zero ✓

Cost J (regulation)  ≈ 12.5808
Cost J (tracking)    ≈ 32.1701


/tmp/ipykernel_3115050/4103715963.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Real-Time Simulation with Agents

synapsys includes a **multi-agent** simulation framework where plant and controller run in **independent threads** synchronised together, communicating through a shared-memory bus with latency < 1 µs.

```
PlantAgent  ──────── y ──────────▶  ControllerAgent
              ◀──── u ────────────
               SharedMemoryTransport
```

Components:
- **`PlantAgent`** — evolves the discrete plant each tick, publishes `y`, consumes `u`.
- **`ControllerAgent`** — reads `y`, computes `u = law(y)`, writes `u`.
- **`SyncEngine`** — manages synchronisation: `WALL_CLOCK` (real time) or `LOCK_STEP` (barrier).
- **`SharedMemoryTransport`** — zero-copy bus (no serialisation, no network).

We use **the same reference plant** and the **same PID controller** from the previous section, now running in separate threads.

In [10]:
import time

from synapsys.agents import ControllerAgent, PlantAgent, SyncEngine, SyncMode
from synapsys.transport import SharedMemoryTransport

DT_AGT = 0.01  # 100 Hz — same DT as PID section
SIM_TIME = 6.0  # seconds
BUS = "nb_demo"
CHANNELS = {"y": 1, "u": 1}

# ── Plant: same G(s) discretised ─────────────────────────────────────────────
plant_agt = c2d(G_ss, dt=DT_AGT, method="zoh")

# ── Shared-memory bus ─────────────────────────────────────────────────────────
owner = SharedMemoryTransport(BUS, CHANNELS, create=True)
owner.write("y", np.array([0.0]))
owner.write("u", np.array([0.0]))

t_plant_h = SharedMemoryTransport(BUS, CHANNELS)
t_ctrl_h = SharedMemoryTransport(BUS, CHANNELS)

# ── PID control law (same gains as previous section) ─────────────────────────
pid_agt = PID(Kp=1.8, Ki=2.5, Kd=0.05, dt=DT_AGT, u_min=-20.0, u_max=20.0)


def law_agt(y: np.ndarray) -> np.ndarray:
    r = 3.0 + 1.5 * np.sin(2 * np.pi * 0.3 * (time.monotonic() - t0_agt))
    return np.array([pid_agt.compute(setpoint=r, measurement=y[0])])


# ── Create agents ─────────────────────────────────────────────────────────────
plant_agent = PlantAgent(
    "plant", plant_agt, t_plant_h, SyncEngine(SyncMode.WALL_CLOCK, dt=DT_AGT)
)
ctrl_agent = ControllerAgent(
    "ctrl", law_agt, t_ctrl_h, SyncEngine(SyncMode.WALL_CLOCK, dt=DT_AGT)
)

# ── Data collection from the owner handle (main thread as monitor) ────────────
log_t, log_y, log_u, log_r = [], [], [], []

t0_agt = time.monotonic()
plant_agent.start(blocking=False)
ctrl_agent.start(blocking=False)

while time.monotonic() - t0_agt < SIM_TIME:
    elapsed = time.monotonic() - t0_agt
    log_t.append(elapsed)
    log_y.append(owner.read("y")[0])
    log_u.append(owner.read("u")[0])
    log_r.append(3.0 + 1.5 * np.sin(2 * np.pi * 0.3 * elapsed))
    time.sleep(DT_AGT)

plant_agent.stop()
ctrl_agent.stop()
t_plant_h.close()
t_ctrl_h.close()
owner.close()

print(f"Collected {len(log_t)} samples over {SIM_TIME}s")
print(f"Final output y = {log_y[-1]:.4f}")

Collected 589 samples over 6.0s
Final output y = 1.8191


In [11]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(
    log_t,
    log_r,
    lw=1.5,
    color="k",
    linestyle="--",
    alpha=0.7,
    label="r(t) — sinusoidal reference",
)
ax1.plot(
    log_t,
    log_y,
    lw=2.0,
    color="steelblue",
    label="y(t) — output (PlantAgent, separate thread)",
)
ax1.set_ylabel("y(t)")
ax1.set_title(
    "Real-Time Simulation — PlantAgent + ControllerAgent (WALL_CLOCK @ 100 Hz)",
    fontsize=12,
)
ax1.legend(fontsize=10)

ax2.plot(
    log_t,
    log_u,
    lw=1.8,
    color="darkorange",
    label="u(t) — PID (ControllerAgent, separate thread)",
)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("u(t)")
ax2.legend(fontsize=10)

fig.tight_layout()
plt.show()

# ── Comparison with manual simulation from Section 6 ─────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    ts, ys, lw=2.5, color="steelblue", label="Section 6 — manual loop (deterministic)"
)
ax.plot(
    log_t,
    log_y,
    lw=1.5,
    color="darkorange",
    linestyle="--",
    label="Section 8 — agents in threads (real-time)",
)
ax.plot(ts, rs, lw=1.2, color="k", linestyle=":", alpha=0.5, label="r(t)")
ax.set_title("Consistency: Manual Loop vs Real-Time Agents", fontsize=13)
ax.set_xlabel("Time (s)")
ax.set_ylabel("y(t)")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print("Both results should be close — small differences are expected")
print("due to OS scheduling jitter in WALL_CLOCK mode.")

Both results should be close — small differences are expected
due to OS scheduling jitter in WALL_CLOCK mode.


/tmp/ipykernel_3115050/424720463.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_3115050/424720463.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9. Conclusion — Comparative Summary

Every section used **the same system**:

$$G(s) = \frac{25}{s^2 + 5s + 25}, \quad \omega_n=5\,\text{rad/s},\; \zeta=0{,}5$$

The table below summarises the numerical results obtained.

In [12]:
# ── Summary table ────────────────────────────────────────────────────────────
print("=" * 65)
print(
    f"  Reference system: G(s) = {WN**2:.0f} / (s² + {2 * ZETA * WN:.0f}s + {WN**2:.0f})"
)
print(f"  ωn={WN} rad/s  |  ζ={ZETA}  |  DC gain={1.0}")
print("=" * 65)

print("\n[Section 2] Representation equivalence:")
print(
    f"  continuous TF  vs  continuous SS   : error = {np.max(np.abs(y_tf - y_ss_i)):.2e}  (machine)"
)
print(
    f"  continuous TF  vs  discrete TF ZOH : error = {np.max(np.abs(y_tf - y_tfd_i)):.5f}  (Δt={DT_CMP}s)"
)
print(
    f"  continuous TF  vs  evolve()         : error = {np.max(np.abs(y_tf - y_man_i)):.5f}  (Δt={DT_CMP}s)"
)

print("\n[Section 3] Discretisation error per method  (Δt=0.05s):")
t_ref2, y_ref2 = step(G_tf)
for method in methods:
    G_d2 = c2d(G_tf, dt=0.05, method=method)
    t_d2, y_d2 = step(G_d2, n=int(3.0 / 0.05) + 1)
    err2 = np.max(np.abs(y_ref2 - np.interp(t_ref2, t_d2, y_d2)))
    print(f"  {method:<15}: {err2:.5f}")

print("\n[Section 4] Steady-state error (proportional controller Kp=2):")
dc_cl = feedback(series(tf([2.0], [1]), G_tf)).evaluate(0).real
print(f"  e_ss = 1 - DC gain = {1 - dc_cl:.4f}  (expected: {1 / (1 + 2 * 1):.4f})")

print("\n[Section 6] PID closed-loop:")
erro_pid = np.array(rs) - np.array(ys)
print(
    f"  RMS error in steady state (t>2s) : {np.sqrt(np.mean(erro_pid[200:] ** 2)):.4f}"
)
print(f"  Peak absolute error              : {np.max(np.abs(erro_pid)):.4f}")

print("\n[Section 7] LQR:")
print(f"  Gain K     = {K}")
print(f"  Nbar       = {Nbar:.4f}")
print(f"  CL poles   : {np.linalg.eigvals(A - B @ K)}")
print(f"  Regulation — final state : x = {x_reg}  (expected: [0, 0])")
print(f"  Tracking   — y_ss = {y_trk_hist[-1]:.6f}  (expected: {SETPOINT_LQR})")

print("\n[Section 8] Real-time agents:")
print(f"  Samples collected : {len(log_t)}")
print(
    f"  Effective rate    : {len(log_t) / SIM_TIME:.1f} Hz  (nominal: {1 / DT_AGT:.0f} Hz)"
)

print()
print("API Summary:")
rows = [
    ("Transfer Function", "tf(num, den)"),
    ("State-Space", "ss(A, B, C, D)"),
    ("Step response", "step(sys) → (t, y)"),
    ("Bode diagram", "bode(sys) → (ω, mag, phase)"),
    ("Closed-loop", "feedback(L)  or  feedback(P, C)"),
    ("Discretisation", "c2d(sys, dt, method)"),
    ("Sample-by-sample sim", "sys.evolve(x, u) → (x_new, y)"),
    ("PID", "PID(Kp, Ki, Kd, dt, u_min, u_max)"),
    ("LQR", "lqr(A, B, Q, R) → (K, P)"),
    ("Distributed simulation", "PlantAgent + ControllerAgent + SharedMemoryTransport"),
]
print(f"  {'Feature':<30} {'API':<50}")
print("  " + "-" * 60)
for feat, api in rows:
    print(f"  {feat:<30} {api:<50}")


  Reference system: G(s) = 25 / (s² + 5s + 25)
  ωn=5.0 rad/s  |  ζ=0.5  |  DC gain=1.0

[Section 2] Representation equivalence:
  continuous TF  vs  continuous SS   : error = 1.55e-15  (machine)
  continuous TF  vs  discrete TF ZOH : error = 0.00020  (Δt=0.01s)
  continuous TF  vs  evolve()         : error = 0.00020  (Δt=0.01s)

[Section 3] Discretisation error per method  (Δt=0.05s):
  zoh            : 0.00668
  bilinear       : 0.06457
  euler          : 0.11678
  backward_diff  : 0.13623

[Section 4] Steady-state error (proportional controller Kp=2):
  e_ss = 1 - DC gain = 0.3333  (expected: 0.3333)

[Section 6] PID closed-loop:
  RMS error in steady state (t>2s) : 0.3493
  Peak absolute error              : 3.0037

[Section 7] LQR:
  Gain K     = [[3.58257569 0.95178386]]
  Nbar       = 4.5826
  CL poles   : [ -4.76828977 -24.02630668]
  Regulation — final state : x = [ 8.26929527e-11 -3.94303960e-10]  (expected: [0, 0])
  Tracking   — y_ss = 2.000000  (expected: 2.0)

[Section 8

---
## 10. Matrix Builders — `mat`, `col`, `row`, `StateEquations`

**`synapsys.utils`** removes the NumPy boilerplate when building $(A, B, C, D)$ matrices.

| Helper | Shape | Use for |
|---|---|---|
| `mat(rows)` | $(m \times n)$ | any 2-D matrix |
| `col(*values)` | $(n \times 1)$ | input matrix **B** |
| `row(*values)` | $(1 \times n)$ | output row selector **C** |

`StateEquations` lets you declare each differential equation **by state name** instead of managing indices manually:

$$\dot{x}_i = \sum_j a_{ij}\,x_j + \sum_k b_{ik}\,u_k$$

Each `.eq(state, **coeffs)` call fills the corresponding row of **A** (state names) and **B** (input names).

In [13]:
import numpy as np

from synapsys.utils import StateEquations, col, mat, row

# ── 2-DOF mass-spring-damper ─────────────────────────────────────────────────
# State vector: [x1, x2, v1, v2],  input: F
# ẋ1 = v1
# ẋ2 = v2
# v̇1 = -2k/m·x1 + k/m·x2 - c/m·v1
# v̇2 =  k/m·x1 - 2k/m·x2           - c/m·v2 + k/m·F

m, c, k = 1.0, 0.1, 2.0

eqs = (
    StateEquations(states=["x1", "x2", "v1", "v2"], inputs=["F"])
    .eq("x1", v1=1)
    .eq("x2", v2=1)
    .eq("v1", x1=-2 * k / m, x2=k / m, v1=-c / m)
    .eq("v2", x1=k / m, x2=-2 * k / m, v2=-c / m, F=k / m)
)

print("A =\n", eqs.A)
print("\nB =\n", eqs.B)

# ── Equivalent hand-built matrices (for comparison) ──────────────────────────
A_manual = mat(
    [
        [0, 0, 1, 0],
        [0, 0, 0, 1],
        [-2 * k / m, k / m, -c / m, 0],
        [k / m, -2 * k / m, 0, -c / m],
    ]
)
B_manual = col(0, 0, 0, k / m)

print(
    "\nMatrices match manual construction:",
    np.allclose(eqs.A, A_manual) and np.allclose(eqs.B, B_manual),
)

# ── Build StateSpace and simulate ────────────────────────────────────────────
from synapsys.api import c2d, ss, step

C = eqs.output("x1", "x2")  # observe positions
D = np.zeros((2, 1))
G_2dof = ss(eqs.A, eqs.B, C, D)

print(f"\nStateSpace: {G_2dof}")
print(f"Stable: {G_2dof.is_stable()}")

# ── Step response of position x1 ─────────────────────────────────────────────
G_d = c2d(G_2dof, dt=0.01)
t, Y = step(G_d, n=800)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, Y[:, 0], label="x1(t) — mass 1 position")
ax.plot(t, Y[:, 1], label="x2(t) — mass 2 position", ls="--")
ax.axhline(Y[-1, 0], color="gray", ls=":", alpha=0.5, label="DC gain")
ax.set_title("2-DOF mass-spring-damper — step response (position outputs)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Position")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# ── row / col demo ────────────────────────────────────────────────────────────
print("\ncol(0, 0, 1).shape =", col(0, 0, 1).shape)  # (3, 1)
print("row(1, 0, 0).shape =", row(1, 0, 0).shape)  # (1, 3)

A =
 [[ 0.   0.   1.   0. ]
 [ 0.   0.   0.   1. ]
 [-4.   2.  -0.1  0. ]
 [ 2.  -4.   0.  -0.1]]

B =
 [[0.]
 [0.]
 [0.]
 [2.]]

Matrices match manual construction: True

StateSpace: StateSpace(n_states=4, n_inputs=1, n_outputs=2, continuous)
Stable: True

col(0, 0, 1).shape = (3, 1)
row(1, 0, 0).shape = (1, 3)


/tmp/ipykernel_3115050/3471689224.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
